# Chapter X: One-vs-Rest Classification Analysis

## Decomposing Regional Discriminability in Brain Connectivity Fingerprinting

---

## Part I: Methodological Rationale

### 1.1 The Progression of Inquiry: From Global to Local Classification

The multinomial classification analysis established that functional connectivity fingerprints reliably identify brain regions. However, this aggregate measure obscures critical variation at the regional level: *which specific regions are most distinguishable? Where do classification boundaries align with—or violate—functional network organization?*

This chapter addresses these questions through **One-vs-Rest (OvR) classification**, which decomposes the multi-class problem into 232 independent binary decisions, each amenable to detailed interrogation.

| Stage | Research Question | Method | Insight |
|-------|------------------|--------|--------|
| **Stage 1** | *Can we distinguish brain regions from connectivity?* | Multinomial LR | Yes, ~92% accuracy |
| **Stage 2** | *Which regions are most/least distinguishable?* | One-vs-Rest | 232 independent classifiers |

### 1.2 Addressing the "Black Box" Problem

| Limitation | Multinomial Behavior | OvR Solution |
|------------|---------------------|---------------|
| **Probability Dilution** | Softmax distributes probability across 232 classes | Independent P(region=k), unbounded by competition |
| **Error Attribution** | Cannot isolate weak fingerprint vs. dominant competitor | Low recall → weak fingerprint; High FP → over-generalization |
| **Feature Interpretation** | Conflates discriminative and suppressive weights | Independent coefficients per classifier |

### 1.3 Theoretical Framework

**Multinomial Logistic Regression:**
$$P(Y=k|X) = \frac{e^{X\beta_k}}{\sum_{j=1}^{K} e^{X\beta_j}}$$

**One-vs-Rest Decomposition:**
$$P(Y=k|X)^{(k)} = \sigma(X\beta^{(k)}) = \frac{1}{1 + e^{-X\beta^{(k)}}}$$

$$\hat{Y} = \arg\max_k P(Y=k|X)^{(k)}$$

---

### Hypotheses

| ID | Null Hypothesis (H₀) | Alternative (H₁) | Statistical Test |
|----|---------------------|------------------|------------------|
| H¹ | Acc(OvR) = Acc(Multi) | Acc(OvR) ≠ Acc(Multi) | McNemar's Test |
| H² | Acc(Rest) = Acc(Task) | Acc(Rest) > Acc(Task) | Paired t-test |
| H³ | Uniform network sensitivity | Differential sensitivity | Kruskal-Wallis |
| H⁴ | Δ(LH) = Δ(RH) | Δ(LH) ≠ Δ(RH) | Mann-Whitney U |
| H⁵ | Error(Rest) = Error(Task) | Patterns differ | Permutation Test |

---

### Analysis Structure

1. Setup & Data Loading
2. Performance Overview & Statistical Comparison (H¹)
3. Per-Region Binary Metrics (Sensitivity, Specificity, AUC)
4. Generalization Gap Analysis (H²)
5. Error Taxonomy Analysis
6. Confidence Calibration
7. Network-Level Deep Dive (9-Panel Heatmaps, H⁵)
8. Hubs of Confusion (Sankey Diagrams)
9. Task-Induced Reorganization (H³, H⁴)
10. Hypothesis Testing Summary
11. Conclusions & Thesis Integration

---
## 1. Setup & Data Loading

In [1]:
# =============================================================================
# LIBRARY IMPORTS
# =============================================================================

# Core Libraries
import numpy as np
import pandas as pd
import json
from pathlib import Path
import warnings
from collections import Counter

# Machine Learning Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, cohen_kappa_score, confusion_matrix
)

# Statistical Analysis
from scipy import stats
from scipy.stats import (
    wilcoxon, ttest_rel, mannwhitneyu, kruskal, 
    fisher_exact, chi2_contingency, binom
)
from statsmodels.stats.multitest import multipletests

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

# Color Palette
COLORS = {
    'ovr': '#e6550d',        # Orange
    'multinomial': '#2b8cbe', # Blue
    'rest': '#31a354',        # Green
    'task': '#de2d26',        # Red
    'positive': '#b2182b',
    'negative': '#2166ac',
    'neutral': '#f7f7f7'
}

print("✓ Libraries imported successfully")
print(f"  NumPy: {np.__version__}")
print(f"  Pandas: {pd.__version__}")

✓ Libraries imported successfully
  NumPy: 2.3.4
  Pandas: 2.3.3


In [2]:
# =============================================================================
# PATH CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path('/home/sjoon/projects/brain_connectivity_classifier')
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'

# Full Model Paths (232 regions)
PATHS = {
    'full_ovr': RESULTS_DIR / 'full_connectivity_analysis' / 'ovr',
    'full_ovr_task': RESULTS_DIR / 'full_connectivity_analysis' / 'task_testing_ovr',
    'full_multi': RESULTS_DIR / 'full_connectivity_analysis' / 'multinomial',
    'full_multi_task': RESULTS_DIR / 'full_connectivity_analysis' / 'task_testing',
    # Left Hemisphere
    'left_ovr': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'ovr',
    'left_ovr_task': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'task_testing_ovr',
    'left_multi': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'multinomial',
    'left_multi_task': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'task_testing_multinomial',
    # Right Hemisphere
    'right_ovr': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'ovr',
    'right_ovr_task': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'task_testing_ovr',
    'right_multi': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'multinomial',
    'right_multi_task': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'task_testing_multinomial',
}

# Verify paths
print("Path Verification:")
for name, path in PATHS.items():
    status = "✓" if path.exists() else "✗"
    print(f"  {status} {name}")

Path Verification:
  ✓ full_ovr
  ✓ full_ovr_task
  ✓ full_multi
  ✓ full_multi_task
  ✓ left_ovr
  ✓ left_ovr_task
  ✓ left_multi
  ✓ left_multi_task
  ✓ right_ovr
  ✓ right_ovr_task
  ✓ right_multi
  ✓ right_multi_task


In [3]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_json(fp):
    """Load JSON file."""
    with open(fp, 'r') as f:
        return json.load(f)

def load_npy(fp):
    """Load numpy file."""
    return np.load(fp, allow_pickle=True)

def load_csv(fp):
    """Load CSV file."""
    return pd.read_csv(fp)

def safe_divide(num, denom, default=0):
    """Safe division with zero handling."""
    return num / denom if denom > 0 else default

class DataLoader:
    """Structured data loader for model results."""
    
    def __init__(self, base_path, is_task=False):
        self.base_path = Path(base_path)
        self.is_task = is_task
        
    def load_cv(self):
        """Load cross-validation results."""
        return {
            'predictions': load_npy(self.base_path / 'cv_predictions.npy'),
            'probabilities': load_npy(self.base_path / 'cv_probabilities.npy'),
            'true_labels': load_npy(self.base_path / 'cv_true_labels.npy'),
            'confusion_matrix': load_npy(self.base_path / 'confusion_matrix.npy'),
            'metrics': load_json(self.base_path / 'overall_metrics.json')
        }
    
    def load_task(self):
        """Load task testing results."""
        return {
            'predictions': load_npy(self.base_path / 'task_predictions.npy'),
            'probabilities': load_npy(self.base_path / 'task_probabilities.npy'),
            'true_labels': load_npy(self.base_path / 'task_true_labels.npy'),
            'confusion_matrix': load_npy(self.base_path / 'task_confusion_matrix.npy'),
            'summary': load_json(self.base_path / 'task_testing_summary.json')
        }

print("✓ Helper functions defined")

✓ Helper functions defined


In [4]:
# =============================================================================
# DATA LOADING
# =============================================================================

print("="*80)
print("LOADING ALL MODEL DATA")
print("="*80)

# Full Model (232 regions)
print("\n[Full Model - 232 regions]")
full_ovr_cv = DataLoader(PATHS['full_ovr']).load_cv()
full_ovr_task = DataLoader(PATHS['full_ovr_task']).load_task()
full_multi_cv = DataLoader(PATHS['full_multi']).load_cv()
full_multi_task = DataLoader(PATHS['full_multi_task']).load_task()
print(f"  OvR CV: {len(full_ovr_cv['predictions']):,} samples")
print(f"  OvR Task: {len(full_ovr_task['predictions']):,} samples")

# Left Hemisphere (116 regions)
print("\n[Left Hemisphere - 116 regions]")
left_ovr_cv = DataLoader(PATHS['left_ovr']).load_cv()
left_ovr_task = DataLoader(PATHS['left_ovr_task']).load_task()
left_multi_cv = DataLoader(PATHS['left_multi']).load_cv()
left_multi_task = DataLoader(PATHS['left_multi_task']).load_task()
print(f"  OvR CV: {len(left_ovr_cv['predictions']):,} samples")

# Right Hemisphere (116 regions)
print("\n[Right Hemisphere - 116 regions]")
right_ovr_cv = DataLoader(PATHS['right_ovr']).load_cv()
right_ovr_task = DataLoader(PATHS['right_ovr_task']).load_task()
right_multi_cv = DataLoader(PATHS['right_multi']).load_cv()
right_multi_task = DataLoader(PATHS['right_multi_task']).load_task()
print(f"  OvR CV: {len(right_ovr_cv['predictions']):,} samples")

# Region Information
region_info = load_csv(PATHS['full_multi'] / 'region_info.csv')
print(f"\n✓ Loaded region info: {len(region_info)} regions")

LOADING ALL MODEL DATA

[Full Model - 232 regions]
  OvR CV: 51,968 samples
  OvR Task: 46,400 samples

[Left Hemisphere - 116 regions]
  OvR CV: 25,984 samples

[Right Hemisphere - 116 regions]
  OvR CV: 25,984 samples

✓ Loaded region info: 232 regions


In [5]:
# =============================================================================
# NETWORK MAPPING & CONSTANTS
# =============================================================================

NETWORK_MAPPING = {
    # Schaefer cortical networks
    'VisCent': 'Visual', 'VisPeri': 'Visual',
    'SomMotA': 'Somatomotor', 'SomMotB': 'Somatomotor',
    'DorsAttnA': 'Dorsal Attention', 'DorsAttnB': 'Dorsal Attention',
    'SalVentAttnA': 'Salience/Ventral Attention', 'SalVentAttnB': 'Salience/Ventral Attention',
    'LimbicA': 'Limbic', 'LimbicB': 'Limbic',
    'ContA': 'Control', 'ContB': 'Control', 'ContC': 'Control',
    'DefaultA': 'Default', 'DefaultB': 'Default', 'DefaultC': 'Default',
    'TempPar': 'Default',
    # Tian subcortical
    'Hippocampus_ant': 'Subcortical', 'Hippocampus_post': 'Subcortical',
    'Amygdala_lat': 'Subcortical', 'Amygdala_med': 'Subcortical',
    'Thalamus_DA': 'Subcortical', 'Thalamus_DP': 'Subcortical',
    'Thalamus_VA': 'Subcortical', 'Thalamus_VP': 'Subcortical',
    'Caudate_ant': 'Subcortical', 'Caudate_post': 'Subcortical',
    'Putamen_ant': 'Subcortical', 'Putamen_post': 'Subcortical',
    'Pallidum_ant': 'Subcortical', 'Pallidum_post': 'Subcortical',
    'Accumbens_core': 'Subcortical', 'Accumbens_shell': 'Subcortical'
}

# Apply mapping
region_info['major_network'] = region_info['network'].map(NETWORK_MAPPING)

# Hemisphere-specific info
region_info_left = region_info[region_info['hemisphere'] == 'left'].reset_index(drop=True)
region_info_right = region_info[region_info['hemisphere'] == 'right'].reset_index(drop=True)

# Constants
NETWORKS = ['Visual', 'Somatomotor', 'Dorsal Attention', 'Salience/Ventral Attention',
            'Limbic', 'Control', 'Default', 'Subcortical']
N_NETWORKS = len(NETWORKS)
N_REGIONS_FULL = 232
N_REGIONS_HEMI = 116

print(f"Networks: {N_NETWORKS}")
print(f"Regions: Full={N_REGIONS_FULL}, Hemisphere={N_REGIONS_HEMI}")
print(f"\nRegions per Network:")
print(region_info['major_network'].value_counts().reindex(NETWORKS))

Networks: 8
Regions: Full=232, Hemisphere=116

Regions per Network:
major_network
Visual                        24
Somatomotor                   34
Dorsal Attention              22
Salience/Ventral Attention    26
Limbic                        14
Control                       37
Default                       43
Subcortical                   32
Name: count, dtype: int64


---
## 2. Performance Overview & Statistical Comparison

### Multi-Level Performance Framework

```
Level 1: Aggregate (Model-Level)
    └── Overall accuracy, Macro F1, Cohen's Kappa
    
Level 2: Network (Functional Systems)
    └── Per-network accuracy, error rates
    
Level 3: Region (Individual Classifiers)
    └── Per-region AUC, Sensitivity, Specificity, PPV, NPV
```

In [6]:
# =============================================================================
# AGGREGATE METRICS
# =============================================================================

def calculate_aggregate_metrics(y_true, y_pred):
    """
    Calculate comprehensive aggregate classification metrics.
    """
    n_samples = len(y_true)
    n_errors = (y_true != y_pred).sum()
    
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'kappa': cohen_kappa_score(y_true, y_pred),
        'n_samples': n_samples,
        'n_errors': n_errors,
        'error_rate': n_errors / n_samples
    }

# Calculate for all models
agg_metrics = {
    'full_ovr_cv': calculate_aggregate_metrics(full_ovr_cv['true_labels'], full_ovr_cv['predictions']),
    'full_ovr_task': calculate_aggregate_metrics(full_ovr_task['true_labels'], full_ovr_task['predictions']),
    'full_multi_cv': calculate_aggregate_metrics(full_multi_cv['true_labels'], full_multi_cv['predictions']),
    'full_multi_task': calculate_aggregate_metrics(full_multi_task['true_labels'], full_multi_task['predictions']),
    'left_ovr_cv': calculate_aggregate_metrics(left_ovr_cv['true_labels'], left_ovr_cv['predictions']),
    'left_ovr_task': calculate_aggregate_metrics(left_ovr_task['true_labels'], left_ovr_task['predictions']),
    'left_multi_cv': calculate_aggregate_metrics(left_multi_cv['true_labels'], left_multi_cv['predictions']),
    'left_multi_task': calculate_aggregate_metrics(left_multi_task['true_labels'], left_multi_task['predictions']),
    'right_ovr_cv': calculate_aggregate_metrics(right_ovr_cv['true_labels'], right_ovr_cv['predictions']),
    'right_ovr_task': calculate_aggregate_metrics(right_ovr_task['true_labels'], right_ovr_task['predictions']),
    'right_multi_cv': calculate_aggregate_metrics(right_multi_cv['true_labels'], right_multi_cv['predictions']),
    'right_multi_task': calculate_aggregate_metrics(right_multi_task['true_labels'], right_multi_task['predictions']),
}

print("✓ Aggregate metrics calculated")

✓ Aggregate metrics calculated


In [7]:
# =============================================================================
# PERFORMANCE COMPARISON TABLE
# =============================================================================

print("="*145)
print("PERFORMANCE COMPARISON: OvR vs MULTINOMIAL")
print("="*145)

comparison_data = []
for model_name, n_regions, prefix in [('Full', 232, 'full'), ('Left', 116, 'left'), ('Right', 116, 'right')]:
    ovr_cv = agg_metrics[f'{prefix}_ovr_cv']
    ovr_task = agg_metrics[f'{prefix}_ovr_task']
    multi_cv = agg_metrics[f'{prefix}_multi_cv']
    multi_task = agg_metrics[f'{prefix}_multi_task']
    
    comparison_data.append({
        'Model': f'{model_name} ({n_regions})',
        'OvR_CV': ovr_cv['accuracy'],
        'OvR_Task': ovr_task['accuracy'],
        'OvR_Drop': ovr_cv['accuracy'] - ovr_task['accuracy'],
        'Multi_CV': multi_cv['accuracy'],
        'Multi_Task': multi_task['accuracy'],
        'Multi_Drop': multi_cv['accuracy'] - multi_task['accuracy'],
        'CV_Diff': ovr_cv['accuracy'] - multi_cv['accuracy'],
        'Task_Diff': ovr_task['accuracy'] - multi_task['accuracy'],
        'OvR_Kappa': ovr_cv['kappa'],
        'Multi_Kappa': multi_cv['kappa']
    })

comparison_df = pd.DataFrame(comparison_data)

print(f"\n{'Model':<15} │ {'OvR CV':>9} {'OvR Task':>10} {'OvR Drop':>10} │ "
      f"{'Multi CV':>9} {'Multi Task':>11} {'Multi Drop':>11} │ {'CV Δ':>8} {'Task Δ':>8} │ {'κ OvR':>7} {'κ Multi':>8}")
print("-"*140)

for _, row in comparison_df.iterrows():
    print(f"{row['Model']:<15} │ {row['OvR_CV']:>8.2%} {row['OvR_Task']:>9.2%} {row['OvR_Drop']:>+9.2%} │ "
          f"{row['Multi_CV']:>8.2%} {row['Multi_Task']:>10.2%} {row['Multi_Drop']:>+10.2%} │ "
          f"{row['CV_Diff']*100:>+7.2f}% {row['Task_Diff']*100:>+7.2f}% │ "
          f"{row['OvR_Kappa']:>6.3f} {row['Multi_Kappa']:>7.3f}")

print("="*145)

PERFORMANCE COMPARISON: OvR vs MULTINOMIAL

Model           │    OvR CV   OvR Task   OvR Drop │  Multi CV  Multi Task  Multi Drop │     CV Δ   Task Δ │   κ OvR  κ Multi
--------------------------------------------------------------------------------------------------------------------------------------------
Full (232)      │   93.17%    89.63%    +3.54% │   92.41%     89.24%     +3.17% │   +0.76%   +0.39% │  0.931   0.924
Left (116)      │   92.26%    87.57%    +4.70% │   91.82%     86.93%     +4.89% │   +0.45%   +0.64% │  0.922   0.917
Right (116)     │   92.08%    87.10%    +4.98% │   91.36%     86.31%     +5.05% │   +0.73%   +0.80% │  0.920   0.913


kappa is (1 - chance accuracy (expected accuracy 1/232))

In [8]:
# =============================================================================
# HYPOTHESIS H¹: McNEMAR'S TEST
# =============================================================================

def mcnemar_test(y_true, pred_multi, pred_ovr):
    """
    McNemar's test for comparing two classifiers.
    
    H₀: Classifiers have equal error rates
    H₁: Classifiers have different error rates
    """
    correct_multi = (pred_multi == y_true)
    correct_ovr = (pred_ovr == y_true)
    
    # b: Multi wrong, OvR correct
    b = np.sum(~correct_multi & correct_ovr)
    # c: Multi correct, OvR wrong
    c = np.sum(correct_multi & ~correct_ovr)
    
    # McNemar with continuity correction
    if b + c > 0:
        statistic = (abs(b - c) - 1)**2 / (b + c)
        p_value = 1 - stats.chi2.cdf(statistic, df=1)
    else:
        statistic, p_value = 0, 1.0
    
    return {
        'b': b, 'c': c,
        'chi2': statistic,
        'p_value': p_value,
        'significant': p_value < 0.05,
        'better': 'OvR' if b > c else ('Multinomial' if c > b else 'Equal')
    }

print("="*115)
print("HYPOTHESIS H¹: McNEMAR'S TEST — OvR vs Multinomial")
print("="*115)
print("\nH₀: OvR and Multinomial achieve equal accuracy")
print("H₁: OvR and Multinomial achieve different accuracy\n")

mcnemar_tests = [
    ('Full (232)', 'CV', full_multi_cv['true_labels'], full_multi_cv['predictions'], full_ovr_cv['predictions']),
    ('Full (232)', 'Task', full_multi_task['true_labels'], full_multi_task['predictions'], full_ovr_task['predictions']),
    ('Left (116)', 'CV', left_multi_cv['true_labels'], left_multi_cv['predictions'], left_ovr_cv['predictions']),
    ('Left (116)', 'Task', left_multi_task['true_labels'], left_multi_task['predictions'], left_ovr_task['predictions']),
    ('Right (116)', 'CV', right_multi_cv['true_labels'], right_multi_cv['predictions'], right_ovr_cv['predictions']),
    ('Right (116)', 'Task', right_multi_task['true_labels'], right_multi_task['predictions'], right_ovr_task['predictions']),
]

mcnemar_results = {}
print(f"{'Model':<15} {'Cond':<6} {'Multi→OvR (b)':>14} {'OvR→Multi (c)':>14} {'χ²':>10} {'p-value':>12} {'Better':>12} {'Sig':>6}")
print("-"*100)

for model, cond, y_true, pred_m, pred_o in mcnemar_tests:
    result = mcnemar_test(y_true, pred_m, pred_o)
    mcnemar_results[f'{model}_{cond}'] = result
    sig = '***' if result['p_value'] < 0.001 else ('**' if result['p_value'] < 0.01 else ('*' if result['p_value'] < 0.05 else 'ns'))
    print(f"{model:<15} {cond:<6} {result['b']:>14,} {result['c']:>14,} {result['chi2']:>10.2f} "
          f"{result['p_value']:>12.2e} {result['better']:>12} {sig:>6}")

print("-"*100)
print("b = Multinomial wrong, OvR correct | c = Multinomial correct, OvR wrong")

HYPOTHESIS H¹: McNEMAR'S TEST — OvR vs Multinomial

H₀: OvR and Multinomial achieve equal accuracy
H₁: OvR and Multinomial achieve different accuracy

Model           Cond    Multi→OvR (b)  OvR→Multi (c)         χ²      p-value       Better    Sig
----------------------------------------------------------------------------------------------------
Full (232)      CV              1,333            939      67.98     1.11e-16          OvR    ***
Full (232)      Task            1,354          1,171      13.12     2.92e-04          OvR    ***
Left (116)      CV                534            418      13.89     1.94e-04          OvR    ***
Left (116)      Task              663            515      18.34     1.84e-05          OvR    ***
Right (116)     CV                567            378      37.40     9.62e-10          OvR    ***
Right (116)     Task              651            466      30.31     3.68e-08          OvR    ***
---------------------------------------------------------------------

\[
\chi^2 = \frac{(b - c)^2}{b + c}
\]

- OvR and Multinomial show very similar overall accuracy across full brain and hemispheric settings, meaning interpretability from OvR is gained without sacrificing performance.  
- Kappa values above 0.9 for both models indicate extremely strong agreement beyond chance, confirming that high accuracy is not just due to class imbalance.  
- McNemar’s test indicates that the OvR models are significantly more accurate than the Multinomial models in all conditions (CV and Task) for Full, Left, and Right regions (p-values ≪ 0.001), confirming that the performance differences are statistically meaningful.


In [9]:
# =============================================================================
# ERROR OVERLAP ANALYSIS
# =============================================================================

def analyze_error_overlap(y_true, pred1, pred2):
    """Analyze overlap between errors from two classifiers."""
    err1 = set(np.where(pred1 != y_true)[0])
    err2 = set(np.where(pred2 != y_true)[0])
    
    shared = len(err1 & err2)
    only1 = len(err1 - err2)
    only2 = len(err2 - err1)
    union = len(err1 | err2)
    
    return {
        'err1': len(err1), 'err2': len(err2),
        'shared': shared, 'only1': only1, 'only2': only2,
        'jaccard': shared / union if union > 0 else 0
    }

print("\n" + "="*105)
print("ERROR OVERLAP ANALYSIS")
print("="*105)
print(f"\n{'Model':<15} {'Cond':<6} {'Multi Err':>10} {'OvR Err':>10} {'Shared':>10} {'Only Multi':>12} {'Only OvR':>10} {'Jaccard':>10}")
print("-"*95)

for model, cond, y_true, pred_m, pred_o in mcnemar_tests:
    result = analyze_error_overlap(y_true, pred_m, pred_o)
    print(f"{model:<15} {cond:<6} {result['err1']:>10,} {result['err2']:>10,} "
          f"{result['shared']:>10,} {result['only1']:>12,} {result['only2']:>10,} "
          f"{result['jaccard']:>9.1%}")

print("="*105)


ERROR OVERLAP ANALYSIS

Model           Cond    Multi Err    OvR Err     Shared   Only Multi   Only OvR    Jaccard
-----------------------------------------------------------------------------------------------
Full (232)      CV          3,943      3,549      2,610        1,333        939     53.5%
Full (232)      Task        4,993      4,810      3,639        1,354      1,171     59.0%
Left (116)      CV          2,126      2,010      1,592          534        418     62.6%
Left (116)      Task        3,032      2,884      2,369          663        515     66.8%
Right (116)     CV          2,246      2,057      1,679          567        378     64.0%
Right (116)     Task        3,177      2,992      2,526          651        466     69.3%


A large portion of errors are shared between Multinomial and OvR models, OvR consistently makes fewer unique errors, and the Jaccard index increases for smaller or task-specific datasets, showing more overlap in challenging samples.

**Example Calculation**
   - For Full (232) CV:
    - Shared errors = 2,610  
    - Only Multi = 1,333  
    - Only OvR = 939  
    - **Union of errors** = 2,610 + 1,333 + 939 = 4,882  
    - **Jaccard** = 2,610 / 4,882 ≈ 53.5%

In [51]:
# =============================================================================
# PERFORMANCE VISUALIZATION
# =============================================================================

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['<b>Cross-Validation (Rest)</b>', '<b>Task Testing (Gender Stroop)</b>']
)

models = ['Full (232)', 'Left (116)', 'Right (116)']

# CV
ovr_cv = [agg_metrics['full_ovr_cv']['accuracy'], agg_metrics['left_ovr_cv']['accuracy'], agg_metrics['right_ovr_cv']['accuracy']]
multi_cv = [agg_metrics['full_multi_cv']['accuracy'], agg_metrics['left_multi_cv']['accuracy'], agg_metrics['right_multi_cv']['accuracy']]

fig.add_trace(go.Bar(name='OvR', x=models, y=ovr_cv, marker_color=COLORS['ovr'],
                     text=[f'{x:.1%}' for x in ovr_cv], textposition='outside'), row=1, col=1)
fig.add_trace(go.Bar(name='Multinomial', x=models, y=multi_cv, marker_color=COLORS['multinomial'],
                     text=[f'{x:.1%}' for x in multi_cv], textposition='outside'), row=1, col=1)

# Task
ovr_task = [agg_metrics['full_ovr_task']['accuracy'], agg_metrics['left_ovr_task']['accuracy'], agg_metrics['right_ovr_task']['accuracy']]
multi_task = [agg_metrics['full_multi_task']['accuracy'], agg_metrics['left_multi_task']['accuracy'], agg_metrics['right_multi_task']['accuracy']]

fig.add_trace(go.Bar(name='OvR', x=models, y=ovr_task, marker_color=COLORS['ovr'],
                     text=[f'{x:.1%}' for x in ovr_task], textposition='outside', showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(name='Multinomial', x=models, y=multi_task, marker_color=COLORS['multinomial'],
                     text=[f'{x:.1%}' for x in multi_task], textposition='outside', showlegend=False), row=1, col=2)

fig.update_layout(
    title=dict(text='<b>Classification Accuracy: OvR vs Multinomial</b>', x=0.5),
    barmode='group', template='simple_white', height=450, width=1000,
    yaxis=dict(range=[0.85, 1.0], tickformat='.0%', title='Accuracy'),
    yaxis2=dict(range=[0.85, 1.0], tickformat='.0%', title='Accuracy'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig.show()

---
## 3. Per-Region Binary Metrics

### OvR-Specific Metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **Sensitivity (TPR)** | $\frac{TP_k}{TP_k + FN_k}$ | How well does classifier *detect* region k? |
| **Specificity (TNR)** | $\frac{TN_k}{TN_k + FP_k}$ | How well does it *reject* non-k regions? |
| **PPV (Precision)** | $\frac{TP_k}{TP_k + FP_k}$ | When predicting k, how often correct? |
| **NPV** (Neg Pred Value)| $\frac{TN_k}{TN_k + FN_k}$ | When rejecting k, how often correct? |
| **AUC-ROC** | Area under ROC | Discrimination across thresholds |

In [11]:
# =============================================================================
# PER-REGION BINARY METRICS CALCULATOR
# =============================================================================

class OvRPerformanceAnalyzer:
    """
    Comprehensive performance analysis for OvR brain connectivity classification.
    """
    
    def __init__(self, n_regions, region_info):
        self.n_regions = n_regions
        self.region_info = region_info
    
    def calculate_binary_metrics(self, y_true, y_pred, y_prob, region_id):
        """
        Calculate all binary metrics for a single region's classifier.
        """
        # Binarize
        y_binary_true = (y_true == region_id).astype(int)
        y_binary_pred = (y_pred == region_id).astype(int)
        region_prob = y_prob[:, region_id]
        
        # Confusion elements
        tp = ((y_binary_pred == 1) & (y_binary_true == 1)).sum()
        tn = ((y_binary_pred == 0) & (y_binary_true == 0)).sum()
        fp = ((y_binary_pred == 1) & (y_binary_true == 0)).sum()
        fn = ((y_binary_pred == 0) & (y_binary_true == 1)).sum()
        
        # Metrics
        sensitivity = safe_divide(tp, tp + fn)
        specificity = safe_divide(tn, tn + fp)
        ppv = safe_divide(tp, tp + fp)
        npv = safe_divide(tn, tn + fn)
        f1 = safe_divide(2 * ppv * sensitivity, ppv + sensitivity)
        
        # AUC
        try:
            auc = roc_auc_score(y_binary_true, region_prob)
        except ValueError:
            auc = 0.5
        
        # Brier
        brier = brier_score_loss(y_binary_true, region_prob)
        
        return {
            'region_id': region_id,
            'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
            'n_positive': tp + fn,
            'sensitivity': sensitivity,
            'specificity': specificity,
            'ppv': ppv,
            'npv': npv,
            'f1': f1,
            'auc_roc': auc,
            'brier_score': brier
        }
    
    def calculate_all_regions(self, y_true, y_pred, y_prob):
        """Calculate metrics for all regions."""
        results = [self.calculate_binary_metrics(y_true, y_pred, y_prob, r) 
                   for r in range(self.n_regions)]
        df = pd.DataFrame(results)
        
        # Merge region info
        df = df.merge(
            self.region_info[['region_idx', 'region_name', 'network', 'major_network', 'hemisphere']],
            left_on='region_id', right_on='region_idx', how='left'
        )
        return df

# Initialize analyzers
analyzer_full = OvRPerformanceAnalyzer(N_REGIONS_FULL, region_info)
analyzer_left = OvRPerformanceAnalyzer(N_REGIONS_HEMI, region_info_left)
analyzer_right = OvRPerformanceAnalyzer(N_REGIONS_HEMI, region_info_right)

print("Calculating per-region binary metrics...")

# Full Model
binary_full_ovr_cv = analyzer_full.calculate_all_regions(
    full_ovr_cv['true_labels'], full_ovr_cv['predictions'], full_ovr_cv['probabilities'])
binary_full_ovr_task = analyzer_full.calculate_all_regions(
    full_ovr_task['true_labels'], full_ovr_task['predictions'], full_ovr_task['probabilities'])
binary_full_multi_cv = analyzer_full.calculate_all_regions(
    full_multi_cv['true_labels'], full_multi_cv['predictions'], full_multi_cv['probabilities'])
binary_full_multi_task = analyzer_full.calculate_all_regions(
    full_multi_task['true_labels'], full_multi_task['predictions'], full_multi_task['probabilities'])

# Left Hemisphere
binary_left_ovr_cv = analyzer_left.calculate_all_regions(
    left_ovr_cv['true_labels'], left_ovr_cv['predictions'], left_ovr_cv['probabilities'])
binary_left_ovr_task = analyzer_left.calculate_all_regions(
    left_ovr_task['true_labels'], left_ovr_task['predictions'], left_ovr_task['probabilities'])

# Right Hemisphere
binary_right_ovr_cv = analyzer_right.calculate_all_regions(
    right_ovr_cv['true_labels'], right_ovr_cv['predictions'], right_ovr_cv['probabilities'])
binary_right_ovr_task = analyzer_right.calculate_all_regions(
    right_ovr_task['true_labels'], right_ovr_task['predictions'], right_ovr_task['probabilities'])

print("✓ Per-region metrics calculated for all models")

Calculating per-region binary metrics...
✓ Per-region metrics calculated for all models


In [55]:
# =============================================================================
# PER-REGION METRICS SUMMARY
# =============================================================================

print("="*100)
print("PER-REGION BINARY METRICS SUMMARY (Full Model, OvR CV)")
print("="*100)

print("\nDescriptive Statistics:")
print(binary_full_ovr_cv[['sensitivity', 'specificity', 'ppv', 'npv', 'auc_roc', 'f1']].describe().round(4))

print("\n" + "-"*85)
print("TOP 10 MOST DISCRIMINABLE REGIONS (by AUC-ROC):")
print("-"*85)
top_auc = binary_full_ovr_cv.nlargest(10, 'auc_roc')
print(f"{'Region':<45} {'Network':<18} {'AUC':>8} {'Sens':>8} {'Spec':>8}")
for _, row in top_auc.iterrows():
    print(f"{row['region_name']:<45} {row['major_network']:<18} {row['auc_roc']:>7.3f} "
          f"{row['sensitivity']:>7.1%} {row['specificity']:>7.3%}")

print("\n" + "-"*85)
print("BOTTOM 10 LEAST DISCRIMINABLE REGIONS (by AUC-ROC):")
print("-"*85)
bottom_auc = binary_full_ovr_cv.nsmallest(10, 'auc_roc')
for _, row in bottom_auc.iterrows():
    print(f"{row['region_name']:<45} {row['major_network']:<18} {row['auc_roc']:>7.3f} "
          f"{row['sensitivity']:>7.1%} {row['specificity']:>7.3%}")

PER-REGION BINARY METRICS SUMMARY (Full Model, OvR CV)

Descriptive Statistics:
       sensitivity  specificity       ppv       npv   auc_roc        f1
count     232.0000     232.0000  232.0000  232.0000  232.0000  232.0000
mean        0.9317       0.9997    0.9332    0.9997    0.9983    0.9322
std         0.0938       0.0005    0.1030    0.0004    0.0051    0.0978
min         0.3795       0.9975    0.4271    0.9973    0.9645    0.4019
25%         0.9241       0.9997    0.9260    0.9997    0.9992    0.9241
50%         0.9598       0.9999    0.9770    0.9998    0.9999    0.9675
75%         0.9821       1.0000    0.9910    0.9999    1.0000    0.9844
max         1.0000       1.0000    1.0000    1.0000    1.0000    1.0000

-------------------------------------------------------------------------------------
TOP 10 MOST DISCRIMINABLE REGIONS (by AUC-ROC):
-------------------------------------------------------------------------------------
Region                                        Netwo

**Key observations**

- The model is very strong overall, with mean AUC ≈ 0.998, F1 ≈ 0.93, and extremely high specificity (TNR) ≈ 0.9997, meaning true negatives are almost always correctly identified.  
- Classification performance varies by region: many cortical regions are perfectly discriminable, while subcortical and limbic areas are harder to classify.  
- The model relies more on cortical, visual, and motor areas for accurate predictions.  
- High specificity (TNR) across regions indicates false positives are rare, but low sensitivity (TPR) in some regions means the model can miss true positives there.  

**Top 10 most discriminable regions**

- These regions have AUC = 1.0, so the classifier can perfectly separate the target class from all others.  
- Most belong to Visual, Somatomotor, Salience/Ventral Attention, Control, and Default networks.  
- Sensitivity and specificity are both very high, close to 100%, so the model detects positives and negatives almost perfectly in these areas.  
- **Interpretation:** These regions are highly informative for the task, and the model relies heavily on them for classification.  

**Bottom 10 least discriminable regions**

- These regions have the lowest AUCs (≈ 0.964–0.990), so the model is less confident distinguishing their activity.  
- They are mostly in Subcortical and Limbic networks, which are harder to classify.  
- Sensitivity (TPR) is relatively low (≈ 37.9–79.5%), while specificity (TNR) remains very high (≈ 99.8–99.9%).  

**Interpretation:** The model rarely mislabels negatives (high specificity) but often misses positives (low sensitivity), so these regions are less informative for classification.  

**Example: low sensitivity, high specificity**

Suppose there are 100 samples for a given brain region:

| Actual / Predicted | Predicted positive | Predicted negative |
|--------------------|--------------------|--------------------|
| Positive (20)      | 8                  | 12                 |
| Negative (80)      | 1                  | 79                 |

**Metrics**

- Sensitivity (True Positive Rate):  
  \(\text{TPR} = \frac{\text{TP}}{\text{TP} + \text{FN}} = \frac{8}{8 + 12} = 0.4\) → low.  
  The model misses 12 out of 20 actual positives.  

- Specificity (True Negative Rate):  
  \(\text{TNR} = \frac{\text{TN}}{\text{TN} + \text{FP}} = \frac{79}{79 + 1} = 0.9875\) → very high.  
  The model almost never mislabels negatives as positive.  

**Interpretation**

- This region provides little useful information for detecting positives, because the model often fails to identify positive cases.  
- However, it is still “safe” in the sense that it does not produce many false alarms, due to high specificity.  
- In the results table, subcortical and limbic regions behave like this: they are hard for the model to recognize as positive, but they almost never cause false positives, making them less informative than the top regions where both sensitivity and specificity are near 100%.

In [13]:
# =============================================================================
# SENSITIVITY VS SPECIFICITY SCATTER
# =============================================================================

fig = px.scatter(
    binary_full_ovr_cv,
    x='specificity', y='sensitivity',
    color='major_network',
    hover_data=['region_name', 'auc_roc', 'f1'],
    opacity=0.7,
    title='<b>Per-Region Sensitivity vs Specificity (Full Model, OvR CV)</b>'
)

fig.add_hline(y=0.9, line_dash='dash', line_color='gray', opacity=0.5)
fig.add_vline(x=0.995, line_dash='dash', line_color='gray', opacity=0.5)

fig.update_layout(
    xaxis_title='Specificity (True Negative Rate)',
    yaxis_title='Sensitivity (True Positive Rate)',
    template='simple_white',
    height=600, width=900,
    xaxis=dict(range=[0.98, 1.001]),
    yaxis=dict(range=[0.5, 1.02])
)
fig.show()

- The model is extremely good at avoiding false positives (specificity ~1 for all regions).
- Detection of true positives varies by region: cortical regions are highly informative, while subcortical/limbic regions are harder to classify correctly.
- This confirms your earlier summary: high specificity everywhere, variable sensitivity by region.

In [14]:
# =============================================================================
# AUC-ROC BY NETWORK
# =============================================================================

fig = px.box(
    binary_full_ovr_cv,
    x='major_network', y='auc_roc',
    color='major_network',
    points='all',
    hover_data=['region_name'],
    title='<b>Per-Region AUC-ROC by Functional Network (Full Model, OvR CV)</b>'
)

fig.add_hline(y=0.9, line_dash='dash', line_color='red', annotation_text='AUC=0.90')
fig.add_hline(y=0.95, line_dash='dot', line_color='green', annotation_text='AUC=0.95')

fig.update_layout(
    xaxis_title='Functional Network',
    yaxis_title='AUC-ROC',
    template='simple_white',
    height=500, width=1000,
    showlegend=False
)
fig.update_xaxes(tickangle=45)
fig.show()

- High AUC in cortical networks → the model relies on these regions for accurate classification.
- Lower AUC in subcortical/limbic networks → these regions provide less informative signal for the task.
- Overall, the plot confirms network-dependent region discriminability, with cortical regions being more informative than subcortical/limbic regions.

---
## 4. Generalization Gap Analysis

### Definition
$$\Delta_{gen}^{(k)} = Acc_{CV}^{(k)} - Acc_{Task}^{(k)}$$

### Interpretation Framework

| Gap | Interpretation | Implication |
|-----|----------------|-------------|
| $\Delta < 0.02$ | Excellent | Fingerprint stable |
| $0.02 \leq \Delta < 0.05$ | Moderate | Some variability |
| $0.05 \leq \Delta < 0.10$ | Poor | Substantial reorganization |
| $\Delta \geq 0.10$ | Failed | Fundamental change |

In [15]:
# =============================================================================
# GENERALIZATION GAP CALCULATION
# =============================================================================

def calculate_generalization_gap(cv_df, task_df):
    """
    Calculate per-region generalization gap.
    """
    gap_df = cv_df[['region_id', 'region_name', 'major_network', 'hemisphere', 
                    'sensitivity', 'auc_roc', 'n_positive']].copy()
    gap_df.columns = ['region_id', 'region_name', 'major_network', 'hemisphere',
                      'cv_sensitivity', 'cv_auc', 'n_samples']
    
    task_cols = task_df[['region_id', 'sensitivity', 'auc_roc']].copy()
    task_cols.columns = ['region_id', 'task_sensitivity', 'task_auc']
    
    gap_df = gap_df.merge(task_cols, on='region_id')
    
    gap_df['sensitivity_gap'] = gap_df['cv_sensitivity'] - gap_df['task_sensitivity']
    gap_df['auc_gap'] = gap_df['cv_auc'] - gap_df['task_auc']
    
    def categorize(gap):
        if gap < 0.02: return 'Excellent'
        elif gap < 0.05: return 'Moderate'
        elif gap < 0.10: return 'Poor'
        else: return 'Failed'
    
    gap_df['gap_category'] = gap_df['sensitivity_gap'].apply(categorize)
    
    return gap_df

# Calculate gaps
gap_full_ovr = calculate_generalization_gap(binary_full_ovr_cv, binary_full_ovr_task)
gap_full_multi = calculate_generalization_gap(binary_full_multi_cv, binary_full_multi_task)
gap_left_ovr = calculate_generalization_gap(binary_left_ovr_cv, binary_left_ovr_task)
gap_right_ovr = calculate_generalization_gap(binary_right_ovr_cv, binary_right_ovr_task)

print("✓ Generalization gaps calculated")

✓ Generalization gaps calculated


In [16]:
# =============================================================================
# GENERALIZATION GAP SUMMARY
# =============================================================================

print("="*100)
print("GENERALIZATION GAP ANALYSIS (Full Model, OvR)")
print("="*100)

print("\nGap Distribution by Category:")
print(gap_full_ovr['gap_category'].value_counts().reindex(['Excellent', 'Moderate', 'Poor', 'Failed']).fillna(0).astype(int))

print("\n" + "-"*85)
print("GAP STATISTICS BY NETWORK:")
print("-"*85)

network_gaps = gap_full_ovr.groupby('major_network').agg({
    'sensitivity_gap': ['mean', 'std', 'min', 'max'],
    'region_id': 'count'
}).round(4)
network_gaps.columns = ['Mean', 'Std', 'Min', 'Max', 'N']
network_gaps = network_gaps.sort_values('Mean', ascending=False)
print(network_gaps)

print("\n" + "-"*85)
print("REGIONS WITH LARGEST GENERALIZATION GAP:")
print("-"*85)
top_gaps = gap_full_ovr.nlargest(15, 'sensitivity_gap')
print(f"{'Region':<45} {'Network':<18} {'CV':>8} {'Task':>8} {'Gap':>8}")
for _, row in top_gaps.iterrows():
    print(f"{row['region_name']:<45} {row['major_network']:<18} {row['cv_sensitivity']:>7.1%} "
          f"{row['task_sensitivity']:>7.1%} {row['sensitivity_gap']:>+7.1%}")

GENERALIZATION GAP ANALYSIS (Full Model, OvR)

Gap Distribution by Category:
gap_category
Excellent    109
Moderate      61
Poor          37
Failed        25
Name: count, dtype: int64

-------------------------------------------------------------------------------------
GAP STATISTICS BY NETWORK:
-------------------------------------------------------------------------------------
                              Mean     Std     Min     Max   N
major_network                                                 
Subcortical                 0.0851  0.0601  0.0071  0.2346  32
Salience/Ventral Attention  0.0426  0.0587 -0.0154  0.2798  26
Limbic                      0.0424  0.0631 -0.0309  0.1943  14
Somatomotor                 0.0294  0.0433 -0.0312  0.1375  34
Control                     0.0258  0.0553 -0.0454  0.1636  37
Visual                      0.0244  0.0250 -0.0045  0.0905  24
Dorsal Attention            0.0205  0.0307 -0.0330  0.0888  22
Default                     0.0184  0.0366 -0.051

In [17]:
# =============================================================================
# HYPOTHESIS H²: REST vs TASK
# =============================================================================

def test_rest_vs_task(cv_df, task_df):
    """
    H₀: CV accuracy = Task accuracy
    H₁: CV accuracy > Task accuracy (one-tailed)
    """
    cv_sens = cv_df['sensitivity'].values
    task_sens = task_df['sensitivity'].values
    
    # Paired t-test
    t_stat, p_two = ttest_rel(cv_sens, task_sens)
    p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2
    
    # Effect size
    diff = cv_sens - task_sens
    cohens_d = diff.mean() / diff.std() if diff.std() > 0 else 0
    
    # Wilcoxon
    w_stat, w_p = wilcoxon(cv_sens, task_sens, alternative='greater')
    
    return {
        'mean_cv': cv_sens.mean(),
        'mean_task': task_sens.mean(),
        'mean_diff': diff.mean(),
        't_stat': t_stat,
        'p_param': p_one,
        'cohens_d': cohens_d,
        'w_stat': w_stat,
        'p_nonparam': w_p
    }

print("="*110)
print("HYPOTHESIS H²: REST vs TASK ACCURACY")
print("="*110)
print("\nH₀: Per-region CV accuracy = Task accuracy")
print("H₁: Per-region CV accuracy > Task accuracy\n")

h2_tests = [
    ('Full OvR', binary_full_ovr_cv, binary_full_ovr_task),
    ('Full Multi', binary_full_multi_cv, binary_full_multi_task),
    ('Left OvR', binary_left_ovr_cv, binary_left_ovr_task),
    ('Right OvR', binary_right_ovr_cv, binary_right_ovr_task),
]

h2_results = {}
print(f"{'Model':<12} {'CV Mean':>10} {'Task Mean':>11} {'Diff':>10} {'t':>8} {'p(t)':>12} {'d':>8} {'p(W)':>12}")
print("-"*95)

for name, cv_df, task_df in h2_tests:
    result = test_rest_vs_task(cv_df, task_df)
    h2_results[name] = result
    sig = '***' if result['p_param'] < 0.001 else ('**' if result['p_param'] < 0.01 else ('*' if result['p_param'] < 0.05 else 'ns'))
    print(f"{name:<12} {result['mean_cv']:>9.2%} {result['mean_task']:>10.2%} {result['mean_diff']:>+9.2%} "
          f"{result['t_stat']:>8.2f} {result['p_param']:>11.2e} {result['cohens_d']:>8.3f} {result['p_nonparam']:>11.2e} {sig}")

print("-"*95)
print("Effect size: |d|<0.2 small, 0.2-0.8 medium, >0.8 large")

HYPOTHESIS H²: REST vs TASK ACCURACY

H₀: Per-region CV accuracy = Task accuracy
H₁: Per-region CV accuracy > Task accuracy

Model           CV Mean   Task Mean       Diff        t         p(t)        d         p(W)
-----------------------------------------------------------------------------------------------
Full OvR        93.17%     89.63%    +3.54%    10.41    2.26e-21    0.685    2.01e-22 ***
Full Multi      92.41%     89.24%    +3.17%     8.79    1.79e-16    0.578    3.10e-15 ***
Left OvR        92.26%     87.57%    +4.70%     8.31    1.09e-13    0.775    4.33e-13 ***
Right OvR       92.08%     87.10%    +4.98%     7.35    1.55e-11    0.686    6.81e-12 ***
-----------------------------------------------------------------------------------------------
Effect size: |d|<0.2 small, 0.2-0.8 medium, >0.8 large


In [18]:
# =============================================================================
# GENERALIZATION GAP DISTRIBUTION PLOT
# =============================================================================

fig = px.box(
    gap_full_ovr,
    x='major_network', y='sensitivity_gap',
    color='major_network',
    points='all',
    hover_data=['region_name'],
    title='<b>Generalization Gap by Network (Full Model, OvR)</b>'
)

fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.add_hline(y=0.05, line_dash='dot', line_color='orange', annotation_text='Moderate')
fig.add_hline(y=0.10, line_dash='dot', line_color='red', annotation_text='Poor')

fig.update_layout(
    xaxis_title='Functional Network',
    yaxis_title='Generalization Gap (CV - Task)',
    template='simple_white',
    height=500, width=1000,
    showlegend=False,
    yaxis=dict(tickformat='.0%')
)
fig.update_xaxes(tickangle=45)
fig.show()

---
## 5. Error Taxonomy Analysis

| Category | Definition |
|----------|------------|
| **Within-Network** | Same hemisphere AND same network |
| **Cross-Network** | Same hemisphere, different network |
| **Cross-Hemisphere** | Different hemisphere, same network |
| **Both Crossed** | Different hemisphere AND network |

In [19]:
# =============================================================================
# ERROR TAXONOMY
# =============================================================================

def analyze_error_taxonomy(y_true, y_pred, reg_info, model_name, condition):
    """
    Categorize errors by hemisphere and network boundaries.
    """
    error_mask = y_true != y_pred
    n_errors = error_mask.sum()
    
    if n_errors == 0:
        return {'Model': model_name, 'Condition': condition, 'N_Errors': 0,
                'Within_Network': 0, 'Cross_Network': 0, 'Cross_Hemisphere': 0, 'Both_Crossed': 0}
    
    lookup = reg_info.set_index('region_idx')
    true_err = y_true[error_mask]
    pred_err = y_pred[error_mask]
    
    true_hemi = np.array([lookup.loc[i, 'hemisphere'] for i in true_err])
    pred_hemi = np.array([lookup.loc[i, 'hemisphere'] for i in pred_err])
    true_net = np.array([lookup.loc[i, 'major_network'] for i in true_err])
    pred_net = np.array([lookup.loc[i, 'major_network'] for i in pred_err])
    
    same_hemi = true_hemi == pred_hemi
    same_net = true_net == pred_net
    
    within = (same_hemi & same_net).sum()
    cross_net = (same_hemi & ~same_net).sum()
    cross_hemi = (~same_hemi & same_net).sum()
    both = (~same_hemi & ~same_net).sum()
    
    return {
        'Model': model_name, 'Condition': condition,
        'N_Errors': n_errors, 'Error_Rate': n_errors / len(y_true),
        'Within_Network': within, 'Cross_Network': cross_net,
        'Cross_Hemisphere': cross_hemi, 'Both_Crossed': both,
        'Within_Pct': within / n_errors * 100,
        'CrossNet_Pct': cross_net / n_errors * 100,
        'CrossHemi_Pct': cross_hemi / n_errors * 100,
        'Both_Pct': both / n_errors * 100
    }

# Analyze
taxonomy_results = [
    analyze_error_taxonomy(full_multi_cv['true_labels'], full_multi_cv['predictions'], region_info, 'Multinomial', 'Rest'),
    analyze_error_taxonomy(full_multi_task['true_labels'], full_multi_task['predictions'], region_info, 'Multinomial', 'Task'),
    analyze_error_taxonomy(full_ovr_cv['true_labels'], full_ovr_cv['predictions'], region_info, 'OvR', 'Rest'),
    analyze_error_taxonomy(full_ovr_task['true_labels'], full_ovr_task['predictions'], region_info, 'OvR', 'Task'),
]

taxonomy_df = pd.DataFrame(taxonomy_results)

print("="*145)
print("ERROR TAXONOMY: Full Model (232 Regions)")
print("="*145)
print(f"\n{'Model':<12} {'Cond':<6} {'Errors':>8} {'Rate':>8} │ {'Within-Net':>16} {'Cross-Net':>16} {'Cross-Hemi':>16} {'Both':>16}")
print("-"*125)

for _, row in taxonomy_df.iterrows():
    print(f"{row['Model']:<12} {row['Condition']:<6} {row['N_Errors']:>8,} {row['Error_Rate']:>7.2%} │ "
          f"{row['Within_Network']:>6} ({row['Within_Pct']:>5.1f}%) "
          f"{row['Cross_Network']:>6} ({row['CrossNet_Pct']:>5.1f}%) "
          f"{row['Cross_Hemisphere']:>6} ({row['CrossHemi_Pct']:>5.1f}%) "
          f"{row['Both_Crossed']:>6} ({row['Both_Pct']:>5.1f}%)")

print("="*145)

ERROR TAXONOMY: Full Model (232 Regions)

Model        Cond     Errors     Rate │       Within-Net        Cross-Net       Cross-Hemi             Both
-----------------------------------------------------------------------------------------------------------------------------
Multinomial  Rest      3,943   7.59% │    592 ( 15.0%)   1319 ( 33.5%)    868 ( 22.0%)   1164 ( 29.5%)
Multinomial  Task      4,993  10.76% │    849 ( 17.0%)   1687 ( 33.8%)   1035 ( 20.7%)   1422 ( 28.5%)
OvR          Rest      3,549   6.83% │    439 ( 12.4%)   1272 ( 35.8%)    704 ( 19.8%)   1134 ( 32.0%)
OvR          Task      4,810  10.37% │    700 ( 14.6%)   1746 ( 36.3%)    865 ( 18.0%)   1499 ( 31.2%)


In [20]:
# =============================================================================
# ERROR TAXONOMY VISUALIZATION
# =============================================================================

categories = ['Within_Pct', 'CrossNet_Pct', 'CrossHemi_Pct', 'Both_Pct']
cat_labels = ['Within Network', 'Cross Network', 'Cross Hemisphere', 'Both Crossed']

fig = make_subplots(rows=1, cols=2, subplot_titles=['<b>Rest (CV)</b>', '<b>Task (Gender Stroop)</b>'])

for col_idx, condition in enumerate(['Rest', 'Task']):
    for model in ['Multinomial', 'OvR']:
        row_data = taxonomy_df[(taxonomy_df['Model'] == model) & (taxonomy_df['Condition'] == condition)].iloc[0]
        values = [row_data[cat] for cat in categories]
        color = COLORS['multinomial'] if model == 'Multinomial' else COLORS['ovr']
        
        fig.add_trace(
            go.Bar(name=model if col_idx == 0 else None, x=cat_labels, y=values, marker_color=color,
                   showlegend=(col_idx == 0), legendgroup=model,
                   text=[f'{v:.1f}%' for v in values], textposition='outside'),
            row=1, col=col_idx+1
        )

fig.update_layout(
    title=dict(text='<b>Error Taxonomy: OvR vs Multinomial</b>', x=0.5),
    barmode='group', template='simple_white', height=450, width=1000,
    yaxis_title='% of Errors', yaxis2_title='% of Errors',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig.update_xaxes(tickangle=30)
fig.show()

---
## 6. Confidence Calibration Analysis

In [57]:
# =============================================================================
# CONFIDENCE ANALYSIS
# =============================================================================

def analyze_confidence(y_true, y_pred, y_prob, n_bins=10):
    """Comprehensive confidence calibration analysis."""
    max_probs = y_prob.max(axis=1)
    correct = y_true == y_pred
    
    correct_conf = max_probs[correct]
    incorrect_conf = max_probs[~correct]
    
    # ECE
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (max_probs >= bin_edges[i]) & (max_probs < bin_edges[i+1])
        if mask.sum() > 0:
            bin_conf = max_probs[mask].mean()
            bin_acc = correct[mask].mean()
            ece += mask.sum() * abs(bin_acc - bin_conf)
    ece /= len(y_true)
    
    return {
        'mean_correct': correct_conf.mean() if len(correct_conf) > 0 else 0,
        'mean_incorrect': incorrect_conf.mean() if len(incorrect_conf) > 0 else 0,
        'conf_gap': correct_conf.mean() - incorrect_conf.mean() if len(incorrect_conf) > 0 else 0,
        'low_conf_correct': (correct_conf < 0.5).mean() * 100 if len(correct_conf) > 0 else 0,
        'high_conf_incorrect': (incorrect_conf > 0.5).mean() * 100 if len(incorrect_conf) > 0 else 0,
        'ece': ece
    }

conf_results = {
    'OvR_Rest': analyze_confidence(full_ovr_cv['true_labels'], full_ovr_cv['predictions'], full_ovr_cv['probabilities']),
    'OvR_Task': analyze_confidence(full_ovr_task['true_labels'], full_ovr_task['predictions'], full_ovr_task['probabilities']),
    'Multi_Rest': analyze_confidence(full_multi_cv['true_labels'], full_multi_cv['predictions'], full_multi_cv['probabilities']),
    'Multi_Task': analyze_confidence(full_multi_task['true_labels'], full_multi_task['predictions'], full_multi_task['probabilities']),
}

print("="*130)
print("CONFIDENCE CALIBRATION ANALYSIS")
print("="*130)
print(f"\n{'Model':<15} {'Correct Conf':>14} {'Incorrect Conf':>16} {'Gap':>10} {'Low-Conf Corr':>15} {'High-Conf Err':>15} {'ECE (Exp calibration Err)':>10}")
print("-"*110)

for name, result in conf_results.items():
    print(f"{name:<15} {result['mean_correct']:>13.1%} {result['mean_incorrect']:>15.1%} "
          f"{result['conf_gap']:>9.1%} {result['low_conf_correct']:>14.1f}% "
          f"{result['high_conf_incorrect']:>14.1f}% {result['ece']:>9.4f}")

print("="*130)

CONFIDENCE CALIBRATION ANALYSIS

Model             Correct Conf   Incorrect Conf        Gap   Low-Conf Corr   High-Conf Err ECE (Exp calibration Err)
--------------------------------------------------------------------------------------------------------------
OvR_Rest                74.8%           42.5%     32.2%           11.2%           32.0%    0.2061
OvR_Task                75.0%           42.8%     32.2%           11.5%           31.1%    0.1795
Multi_Rest              87.1%           36.9%     50.2%            7.3%           23.2%    0.0913
Multi_Task              84.7%           38.0%     46.7%            9.3%           25.2%    0.0954


In [22]:
# =============================================================================
# CONFIDENCE DISTRIBUTIONS
# =============================================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['<b>Multinomial - Rest</b>', '<b>Multinomial - Task</b>',
                    '<b>OvR - Rest</b>', '<b>OvR - Task</b>'],
    vertical_spacing=0.12
)

configs = [
    (full_multi_cv, 1, 1), (full_multi_task, 1, 2),
    (full_ovr_cv, 2, 1), (full_ovr_task, 2, 2)
]

for data, row, col in configs:
    max_probs = data['probabilities'].max(axis=1)
    correct = data['true_labels'] == data['predictions']
    
    fig.add_trace(go.Histogram(x=max_probs[correct], name='Correct', opacity=0.7,
                               marker_color=COLORS['multinomial'], nbinsx=50, histnorm='probability',
                               showlegend=(row==1 and col==1)), row=row, col=col)
    fig.add_trace(go.Histogram(x=max_probs[~correct], name='Incorrect', opacity=0.7,
                               marker_color=COLORS['ovr'], nbinsx=50, histnorm='probability',
                               showlegend=(row==1 and col==1)), row=row, col=col)

fig.update_layout(
    title=dict(text='<b>Prediction Confidence Distributions</b>', x=0.5),
    barmode='overlay', template='simple_white', height=600, width=1000,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig.update_xaxes(title_text='Max Probability', range=[0, 1])
fig.update_yaxes(title_text='Proportion')
fig.show()

---
## 7. Network-Level Deep Dive (9-Panel Heatmaps)

### 3×3 Grid Structure
```
                Rest          Task          Δ(Task-Rest)
Full (232)  [Heatmap 1]   [Heatmap 2]    [Heatmap 3]
Left (116)  [Heatmap 4]   [Heatmap 5]    [Heatmap 6]  
Right(116)  [Heatmap 7]   [Heatmap 8]    [Heatmap 9]
```

In [23]:
# =============================================================================
# NETWORK AGGREGATION
# =============================================================================

def aggregate_cm_to_network(cm, reg_info, networks):
    """Aggregate confusion matrix to network level."""
    net_labels = reg_info['major_network'].values
    df = pd.DataFrame(cm, index=net_labels, columns=net_labels)
    row_grouped = df.groupby(level=0).sum()
    full_grouped = row_grouped.T.groupby(level=0).sum().T
    return full_grouped.reindex(index=networks, columns=networks).fillna(0).values.astype(int)

# Aggregate all
net_full_ovr_cv = aggregate_cm_to_network(full_ovr_cv['confusion_matrix'], region_info, NETWORKS)
net_full_ovr_task = aggregate_cm_to_network(full_ovr_task['confusion_matrix'], region_info, NETWORKS)
net_left_ovr_cv = aggregate_cm_to_network(left_ovr_cv['confusion_matrix'], region_info_left, NETWORKS)
net_left_ovr_task = aggregate_cm_to_network(left_ovr_task['confusion_matrix'], region_info_left, NETWORKS)
net_right_ovr_cv = aggregate_cm_to_network(right_ovr_cv['confusion_matrix'], region_info_right, NETWORKS)
net_right_ovr_task = aggregate_cm_to_network(right_ovr_task['confusion_matrix'], region_info_right, NETWORKS)
net_full_multi_cv = aggregate_cm_to_network(full_multi_cv['confusion_matrix'], region_info, NETWORKS)
net_full_multi_task = aggregate_cm_to_network(full_multi_task['confusion_matrix'], region_info, NETWORKS)

print("✓ Network-level confusion matrices created")

✓ Network-level confusion matrices created


In [24]:
# =============================================================================
# 9-PANEL NETWORK HEATMAP
# =============================================================================

fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=['<b>Full: Rest</b>', '<b>Full: Task</b>', '<b>Full: Δ</b>',
                    '<b>Left: Rest</b>', '<b>Left: Task</b>', '<b>Left: Δ</b>',
                    '<b>Right: Rest</b>', '<b>Right: Task</b>', '<b>Right: Δ</b>'],
    horizontal_spacing=0.06, vertical_spacing=0.08
)

seq_cs = [[0, '#f7fbff'], [0.5, '#6baed6'], [1, '#08306b']]
div_cs = [[0, '#2166ac'], [0.5, '#f7f7f7'], [1, '#b2182b']]

configs = [
    (net_full_ovr_cv, net_full_ovr_task, 1),
    (net_left_ovr_cv, net_left_ovr_task, 2),
    (net_right_ovr_cv, net_right_ovr_task, 3)
]

for rest_cm, task_cm, row in configs:
    # Rest
    rest_m = rest_cm.astype(float).copy()
    np.fill_diagonal(rest_m, np.nan)
    fig.add_trace(go.Heatmap(z=rest_m, x=NETWORKS, y=NETWORKS, colorscale=seq_cs, showscale=False), row=row, col=1)
    
    # Task
    task_m = task_cm.astype(float).copy()
    np.fill_diagonal(task_m, np.nan)
    fig.add_trace(go.Heatmap(z=task_m, x=NETWORKS, y=NETWORKS, colorscale=seq_cs, showscale=False), row=row, col=2)
    
    # Difference
    diff = (task_cm - rest_cm).astype(float)
    np.fill_diagonal(diff, np.nan)
    limit = np.nanmax(np.abs(diff))
    fig.add_trace(go.Heatmap(z=diff, x=NETWORKS, y=NETWORKS, colorscale=div_cs, zmin=-limit, zmax=limit, showscale=(row==1)), row=row, col=3)

fig.update_layout(
    title=dict(text='<b>Network-Level Confusion: 9-Panel Framework (OvR)</b>', x=0.5),
    template='plotly_white', height=900, width=1200
)
fig.update_xaxes(tickangle=45, tickfont=dict(size=7))
fig.update_yaxes(autorange='reversed', tickfont=dict(size=7))
fig.show()

In [25]:
# =============================================================================
# HYPOTHESIS H⁵: PERMUTATION TEST
# =============================================================================

def permutation_test_cm_diff(cm_rest, cm_task, n_perm=10000):
    """Permutation test for Rest-Task difference."""
    mask = ~np.eye(len(NETWORKS), dtype=bool)
    observed = np.abs(cm_task[mask] - cm_rest[mask]).sum()
    
    combined = np.stack([cm_rest, cm_task])
    null_dist = []
    for _ in range(n_perm):
        idx = np.random.permutation(2)
        perm_diff = np.abs(combined[idx[1]][mask] - combined[idx[0]][mask]).sum()
        null_dist.append(perm_diff)
    
    p_value = np.mean(np.array(null_dist) >= observed)
    return observed, p_value

print("="*80)
print("HYPOTHESIS H⁵: PERMUTATION TEST — Rest vs Task Patterns")
print("="*80)
print("\nH₀: Network error patterns don't differ between Rest and Task")
print("H₁: Task induces systematic changes\n")

perm_tests = [
    ('Full OvR', net_full_ovr_cv, net_full_ovr_task),
    ('Full Multi', net_full_multi_cv, net_full_multi_task),
    ('Left OvR', net_left_ovr_cv, net_left_ovr_task),
    ('Right OvR', net_right_ovr_cv, net_right_ovr_task),
]

print(f"{'Model':<15} {'Observed':>12} {'p-value':>12} {'Significant':>12}")
print("-"*55)

perm_results = {}
for name, rest, task in perm_tests:
    obs, p = permutation_test_cm_diff(rest, task)
    perm_results[name] = {'observed': obs, 'p_value': p}
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f"{name:<15} {obs:>12.0f} {p:>12.4f} {sig:>12}")

HYPOTHESIS H⁵: PERMUTATION TEST — Rest vs Task Patterns

H₀: Network error patterns don't differ between Rest and Task
H₁: Task induces systematic changes

Model               Observed      p-value  Significant
-------------------------------------------------------
Full OvR                 933       1.0000           ns
Full Multi               852       1.0000           ns
Left OvR                 731       1.0000           ns
Right OvR                733       1.0000           ns


---
## 8. Hubs of Confusion Analysis

In [26]:
# =============================================================================
# CONFUSION HUBS
# =============================================================================

def identify_hubs(binary_df, top_n=15):
    """Identify source and sink hubs."""
    df = binary_df.copy()
    df['source_score'] = 1 - df['sensitivity']
    df['sink_score'] = 1 - df['specificity']
    
    source = df.nlargest(top_n, 'source_score')[['region_name', 'major_network', 'sensitivity', 'source_score', 'fn']]
    sink = df.nlargest(top_n, 'sink_score')[['region_name', 'major_network', 'specificity', 'sink_score', 'fp']]
    
    return source, sink

source_hubs, sink_hubs = identify_hubs(binary_full_ovr_cv)

print("="*95)
print("HUBS OF CONFUSION (Full Model, OvR CV)")
print("="*95)

print("\nSOURCE HUBS (Low Sensitivity — frequently misclassified):")
print("-"*80)
print(f"{'Region':<45} {'Network':<18} {'Sens':>8} {'FN':>6}")
for _, row in source_hubs.iterrows():
    print(f"{row['region_name']:<45} {row['major_network']:<18} {row['sensitivity']:>7.1%} {row['fn']:>6}")

print("\nSINK HUBS (Low Specificity — attract misclassifications):")
print("-"*80)
print(f"{'Region':<45} {'Network':<18} {'Spec':>8} {'FP':>6}")
for _, row in sink_hubs.iterrows():
    print(f"{row['region_name']:<45} {row['major_network']:<18} {row['specificity']:>7.3%} {row['fp']:>6}")

HUBS OF CONFUSION (Full Model, OvR CV)

SOURCE HUBS (Low Sensitivity — frequently misclassified):
--------------------------------------------------------------------------------
Region                                        Network                Sens     FN
aGP-rh                                        Subcortical          37.9%    139
aGP-lh                                        Subcortical          42.0%    130
pGP-lh                                        Subcortical          43.3%    127
pGP-rh                                        Subcortical          46.9%    119
RH_DefaultA_PFCm_2                            Default              55.4%    100
LH_LimbicA_TempPole_4                         Limbic               64.7%     79
RH_LimbicA_TempPole_3                         Limbic               70.1%     67
lAMY-rh                                       Subcortical          71.9%     63
THA-DP-lh                                     Subcortical          73.2%     60
lAMY-lh             

In [27]:
# =============================================================================
# CONFUSION FLOW & SANKEY
# =============================================================================

def build_confusion_flow(y_true, y_pred, reg_info, min_count=3):
    """Build confusion flow matrix."""
    error_mask = y_true != y_pred
    counts = Counter(zip(y_true[error_mask], y_pred[error_mask]))
    lookup = reg_info.set_index('region_idx')
    
    flows = []
    for (t, p), c in counts.items():
        if c >= min_count:
            flows.append({
                'true_network': lookup.loc[t, 'major_network'],
                'pred_network': lookup.loc[p, 'major_network'],
                'count': c
            })
    return pd.DataFrame(flows)

flow_df = build_confusion_flow(full_ovr_cv['true_labels'], full_ovr_cv['predictions'], region_info)
net_flow = flow_df.groupby(['true_network', 'pred_network'])['count'].sum().reset_index()
net_flow = net_flow[net_flow['true_network'] != net_flow['pred_network']]

# Sankey
left_nodes = [f"True: {n}" for n in NETWORKS]
right_nodes = [f"Pred: {n}" for n in NETWORKS]
all_nodes = left_nodes + right_nodes
node_idx = {n: i for i, n in enumerate(all_nodes)}

colors = px.colors.qualitative.Set2[:len(NETWORKS)]
net_colors = dict(zip(NETWORKS, colors))

sources, targets, values, link_colors = [], [], [], []
for _, row in net_flow.iterrows():
    sources.append(node_idx[f"True: {row['true_network']}"])
    targets.append(node_idx[f"Pred: {row['pred_network']}"])
    values.append(row['count'])
    link_colors.append(net_colors[row['true_network']].replace('rgb', 'rgba').replace(')', ', 0.5)'))

fig = go.Figure(go.Sankey(
    node=dict(pad=15, thickness=20, label=all_nodes,
              color=[net_colors.get(n.split(': ')[1], 'gray') for n in all_nodes]),
    link=dict(source=sources, target=targets, value=values, color=link_colors)
))

fig.update_layout(
    title=dict(text='<b>Network Error Flow (OvR CV)</b>', x=0.5),
    height=600, width=1000
)
fig.show()

---
## 9. Task-Induced Reorganization (H³, H⁴)

In [28]:
# =============================================================================
# HYPOTHESIS H³: NETWORK DIFFERENTIAL SENSITIVITY
# =============================================================================

print("="*100)
print("HYPOTHESIS H³: NETWORK DIFFERENTIAL SENSITIVITY")
print("="*100)
print("\nH₀: Task-induced changes uniform across networks")
print("H₁: At least one network shows differential sensitivity\n")

# Kruskal-Wallis
groups = [grp['sensitivity_gap'].values for _, grp in gap_full_ovr.groupby('major_network')]
h_stat, h_p = kruskal(*groups)

n = len(gap_full_ovr)
k = len(NETWORKS)
eta_sq = (h_stat - k + 1) / (n - k) if n > k else 0

print(f"Kruskal-Wallis H: {h_stat:.2f}")
print(f"p-value: {h_p:.2e}")
print(f"Effect size (η²): {eta_sq:.4f}")
print(f"\nConclusion: {'Reject H₀ — Networks differ' if h_p < 0.05 else 'Fail to reject H₀'}")

# Network sensitivity summary
net_sens = gap_full_ovr.groupby('major_network').agg({
    'cv_sensitivity': 'mean', 'task_sensitivity': 'mean', 'sensitivity_gap': 'mean'
}).round(4).sort_values('sensitivity_gap', ascending=False)

print("\n" + "-"*70)
print("NETWORK SENSITIVITY RANKING:")
print("-"*70)
print(f"{'Network':<28} {'CV Sens':>10} {'Task Sens':>11} {'Gap':>10}")
for net, row in net_sens.iterrows():
    print(f"{net:<28} {row['cv_sensitivity']:>9.1%} {row['task_sensitivity']:>10.1%} {row['sensitivity_gap']:>+9.1%}")

HYPOTHESIS H³: NETWORK DIFFERENTIAL SENSITIVITY

H₀: Task-induced changes uniform across networks
H₁: At least one network shows differential sensitivity

Kruskal-Wallis H: 37.50
p-value: 3.78e-06
Effect size (η²): 0.1361

Conclusion: Reject H₀ — Networks differ

----------------------------------------------------------------------
NETWORK SENSITIVITY RANKING:
----------------------------------------------------------------------
Network                         CV Sens   Task Sens        Gap
Subcortical                      82.1%      73.6%     +8.5%
Salience/Ventral Attention       94.6%      90.4%     +4.3%
Limbic                           85.5%      81.2%     +4.2%
Somatomotor                      96.8%      93.8%     +2.9%
Control                          95.4%      92.8%     +2.6%
Visual                           97.5%      95.1%     +2.4%
Dorsal Attention                 96.0%      93.9%     +2.1%
Default                          94.4%      92.5%     +1.8%


In [29]:
# =============================================================================
# HYPOTHESIS H⁴: HEMISPHERIC ASYMMETRY
# =============================================================================

print("\n" + "="*100)
print("HYPOTHESIS H⁴: HEMISPHERIC ASYMMETRY")
print("="*100)
print("\nH₀: Left and right hemispheres show equal task sensitivity")
print("H₁: Hemispheres show asymmetric sensitivity\n")

lh_gap = gap_full_ovr[gap_full_ovr['hemisphere'] == 'left']['sensitivity_gap'].values
rh_gap = gap_full_ovr[gap_full_ovr['hemisphere'] == 'right']['sensitivity_gap'].values

u_stat, u_p = mannwhitneyu(lh_gap, rh_gap, alternative='two-sided')
r_effect = 1 - (2 * u_stat) / (len(lh_gap) * len(rh_gap))

print(f"Left Hemisphere:  Mean Gap = {lh_gap.mean():+.4f} (SD = {lh_gap.std():.4f})")
print(f"Right Hemisphere: Mean Gap = {rh_gap.mean():+.4f} (SD = {rh_gap.std():.4f})")
print(f"Difference (LH - RH): {lh_gap.mean() - rh_gap.mean():+.4f}")
print(f"\nMann-Whitney U: {u_stat:.2f}")
print(f"p-value: {u_p:.4f}")
print(f"Effect size (r): {r_effect:.4f}")
print(f"\nConclusion: {'Reject H₀ — Asymmetry exists' if u_p < 0.05 else 'Fail to reject H₀'}")


HYPOTHESIS H⁴: HEMISPHERIC ASYMMETRY

H₀: Left and right hemispheres show equal task sensitivity
H₁: Hemispheres show asymmetric sensitivity

Left Hemisphere:  Mean Gap = +0.0306 (SD = 0.0447)
Right Hemisphere: Mean Gap = +0.0402 (SD = 0.0573)
Difference (LH - RH): -0.0096

Mann-Whitney U: 6118.50
p-value: 0.2335
Effect size (r): 0.0906

Conclusion: Fail to reject H₀


In [30]:
# =============================================================================
# VOLCANO PLOT
# =============================================================================

# Significance per region
def binomial_p(cv, task, n):
    if n == 0 or cv == 0:
        return 1.0
    k = int(task * n)
    return binom.cdf(k, n, cv)

gap_full_ovr['p_value'] = gap_full_ovr.apply(
    lambda r: binomial_p(r['cv_sensitivity'], r['task_sensitivity'], int(r['n_samples'])), axis=1)
gap_full_ovr['neg_log_p'] = -np.log10(gap_full_ovr['p_value'].clip(lower=1e-10))
gap_full_ovr['significant'] = gap_full_ovr['p_value'] < 0.05

fig = px.scatter(
    gap_full_ovr, x='sensitivity_gap', y='neg_log_p',
    color='major_network', hover_data=['region_name'],
    opacity=0.7, title='<b>Task-Induced Reorganization: Volcano Plot</b>'
)

fig.add_hline(y=-np.log10(0.05), line_dash='dash', line_color='red', annotation_text='p=0.05')

fig.update_layout(
    xaxis_title='Accuracy Drop (CV - Task)',
    yaxis_title='-log₁₀(p)',
    template='simple_white',
    height=600, width=1000,
    xaxis=dict(tickformat='.0%')
)
fig.show()

print(f"\nSignificant regions: {gap_full_ovr['significant'].sum()} / {len(gap_full_ovr)}")


Significant regions: 127 / 232


In [31]:
# =============================================================================
# TASK SENSITIVITY STRIP PLOT
# =============================================================================

fig = px.strip(
    gap_full_ovr, x='major_network', y='sensitivity_gap',
    color='major_network', hover_data=['region_name'],
    title='<b>Task-Induced Accuracy Drop by Network</b>'
)

# Add means
for net in NETWORKS:
    mean = gap_full_ovr[gap_full_ovr['major_network'] == net]['sensitivity_gap'].mean()
    fig.add_trace(go.Scatter(
        x=[net], y=[mean], mode='markers',
        marker=dict(symbol='diamond', size=15, color='black', line=dict(width=2, color='white')),
        showlegend=False
    ))

fig.add_hline(y=0, line_dash='dash', line_color='gray')

fig.update_layout(
    xaxis_title='Network', yaxis_title='Accuracy Drop',
    template='simple_white', height=550, width=1000,
    showlegend=False, yaxis=dict(tickformat='.0%')
)
fig.update_xaxes(tickangle=45)
fig.show()

---
## 10. Hypothesis Testing Summary

In [32]:
# =============================================================================
# HYPOTHESIS TESTING SUMMARY TABLE
# =============================================================================

print("="*150)
print("HYPOTHESIS TESTING SUMMARY")
print("="*150)

print("""
┌────────┬───────────────────────────────────────────────────────┬────────────────────┬──────────────┬──────────────┐
│   ID   │                      Hypothesis                        │        Test        │   p-value    │    Result    │
├────────┼───────────────────────────────────────────────────────┼────────────────────┼──────────────┼──────────────┤""")

# H1
h1_p = mcnemar_results['Full (232)_CV']['p_value']
h1_r = 'Reject H₀' if h1_p < 0.05 else 'Fail to reject'
print(f"│   H¹   │ OvR accuracy = Multinomial accuracy                   │ McNemar's          │ {h1_p:>10.2e} │ {h1_r:<12} │")

# H2
h2_p = h2_results['Full OvR']['p_param']
h2_r = 'Reject H₀' if h2_p < 0.05 else 'Fail to reject'
print(f"│   H²   │ CV accuracy = Task accuracy                           │ Paired t-test      │ {h2_p:>10.2e} │ {h2_r:<12} │")

# H3
h3_r = 'Reject H₀' if h_p < 0.05 else 'Fail to reject'
print(f"│   H³   │ Uniform network sensitivity                           │ Kruskal-Wallis     │ {h_p:>10.2e} │ {h3_r:<12} │")

# H4
h4_r = 'Reject H₀' if u_p < 0.05 else 'Fail to reject'
print(f"│   H⁴   │ LH sensitivity = RH sensitivity                       │ Mann-Whitney U     │ {u_p:>10.4f} │ {h4_r:<12} │")

# H5
h5_p = perm_results['Full OvR']['p_value']
h5_r = 'Reject H₀' if h5_p < 0.05 else 'Fail to reject'
print(f"│   H⁵   │ Rest error patterns = Task error patterns             │ Permutation        │ {h5_p:>10.4f} │ {h5_r:<12} │")

print("└────────┴───────────────────────────────────────────────────────┴────────────────────┴──────────────┴──────────────┘")

HYPOTHESIS TESTING SUMMARY

┌────────┬───────────────────────────────────────────────────────┬────────────────────┬──────────────┬──────────────┐
│   ID   │                      Hypothesis                        │        Test        │   p-value    │    Result    │
├────────┼───────────────────────────────────────────────────────┼────────────────────┼──────────────┼──────────────┤
│   H¹   │ OvR accuracy = Multinomial accuracy                   │ McNemar's          │   1.11e-16 │ Reject H₀    │
│   H²   │ CV accuracy = Task accuracy                           │ Paired t-test      │   2.26e-21 │ Reject H₀    │
│   H³   │ Uniform network sensitivity                           │ Kruskal-Wallis     │   3.78e-06 │ Reject H₀    │
│   H⁴   │ LH sensitivity = RH sensitivity                       │ Mann-Whitney U     │     0.2335 │ Fail to reject │
│   H⁵   │ Rest error patterns = Task error patterns             │ Permutation        │     1.0000 │ Fail to reject │
└────────┴───────────────────────

---
## 11. Conclusions & Thesis Integration

In [33]:
# =============================================================================
# COMPREHENSIVE SUMMARY
# =============================================================================

print("="*140)
print("COMPREHENSIVE ANALYSIS SUMMARY")
print("="*140)

# Taxonomy comparison
ovr_within = taxonomy_df[(taxonomy_df['Model']=='OvR') & (taxonomy_df['Condition']=='Rest')]['Within_Pct'].iloc[0]
multi_within = taxonomy_df[(taxonomy_df['Model']=='Multinomial') & (taxonomy_df['Condition']=='Rest')]['Within_Pct'].iloc[0]

print(f"""
1. PERFORMANCE COMPARISON
   ─────────────────────────────────────────────────────────────────────────
   Full Model (232 regions):
   ├── OvR CV Accuracy:        {agg_metrics['full_ovr_cv']['accuracy']:.2%}
   ├── Multinomial CV Acc:     {agg_metrics['full_multi_cv']['accuracy']:.2%}
   ├── Difference (CV):        {(agg_metrics['full_ovr_cv']['accuracy'] - agg_metrics['full_multi_cv']['accuracy'])*100:+.2f}%
   ├── McNemar p-value:        {mcnemar_results['Full (232)_CV']['p_value']:.2e}
   └── Generalization Gap:     {(agg_metrics['full_ovr_cv']['accuracy'] - agg_metrics['full_ovr_task']['accuracy'])*100:+.2f}%

2. ERROR TAXONOMY
   ─────────────────────────────────────────────────────────────────────────
   Within-Network Errors (Rest):
   ├── OvR:          {ovr_within:.1f}%
   ├── Multinomial:  {multi_within:.1f}%
   └── Difference:   {ovr_within - multi_within:+.1f}%

3. CONFIDENCE CALIBRATION
   ─────────────────────────────────────────────────────────────────────────
   ECE:              OvR = {conf_results['OvR_Rest']['ece']:.4f}, Multi = {conf_results['Multi_Rest']['ece']:.4f}
   High-Conf Errors: OvR = {conf_results['OvR_Rest']['high_conf_incorrect']:.1f}%, Multi = {conf_results['Multi_Rest']['high_conf_incorrect']:.1f}%

4. TASK-INDUCED REORGANIZATION
   ─────────────────────────────────────────────────────────────────────────
   Most sensitive network:  {net_sens.index[0]} (Gap: {net_sens.iloc[0]['sensitivity_gap']:+.1%})
   Least sensitive network: {net_sens.index[-1]} (Gap: {net_sens.iloc[-1]['sensitivity_gap']:+.1%})
   Significant regions:     {gap_full_ovr['significant'].sum()} / {len(gap_full_ovr)} ({gap_full_ovr['significant'].mean()*100:.1f}%)

5. KEY FINDINGS
   ─────────────────────────────────────────────────────────────────────────
   • OvR decomposes classification into 232 interpretable binary decisions
   • Per-region metrics reveal which brain regions are most/least discriminable
   • Task engagement induces differential sensitivity across functional networks
   • The "error-as-signal" framework is supported by systematic network patterns
""")

print("="*140)

COMPREHENSIVE ANALYSIS SUMMARY

1. PERFORMANCE COMPARISON
   ─────────────────────────────────────────────────────────────────────────
   Full Model (232 regions):
   ├── OvR CV Accuracy:        93.17%
   ├── Multinomial CV Acc:     92.41%
   ├── Difference (CV):        +0.76%
   ├── McNemar p-value:        1.11e-16
   └── Generalization Gap:     +3.54%

2. ERROR TAXONOMY
   ─────────────────────────────────────────────────────────────────────────
   Within-Network Errors (Rest):
   ├── OvR:          12.4%
   ├── Multinomial:  15.0%
   └── Difference:   -2.6%

3. CONFIDENCE CALIBRATION
   ─────────────────────────────────────────────────────────────────────────
   ECE:              OvR = 0.2061, Multi = 0.0913
   High-Conf Errors: OvR = 32.0%, Multi = 23.2%

4. TASK-INDUCED REORGANIZATION
   ─────────────────────────────────────────────────────────────────────────
   Most sensitive network:  Subcortical (Gap: +8.5%)
   Least sensitive network: Default (Gap: +1.8%)
   Significant region

In [34]:
# =============================================================================
# THESIS SUMMARY TABLE
# =============================================================================

summary_df = pd.DataFrame({
    'Metric': [
        'CV Accuracy', 'Task Accuracy', 'Generalization Gap',
        'Within-Network Errors', 'ECE', 'High-Confidence Errors',
        'Mean Per-Region AUC'
    ],
    'OvR': [
        f"{agg_metrics['full_ovr_cv']['accuracy']:.2%}",
        f"{agg_metrics['full_ovr_task']['accuracy']:.2%}",
        f"{(agg_metrics['full_ovr_cv']['accuracy'] - agg_metrics['full_ovr_task']['accuracy']):.2%}",
        f"{ovr_within:.1f}%",
        f"{conf_results['OvR_Rest']['ece']:.4f}",
        f"{conf_results['OvR_Rest']['high_conf_incorrect']:.1f}%",
        f"{binary_full_ovr_cv['auc_roc'].mean():.4f}"
    ],
    'Multinomial': [
        f"{agg_metrics['full_multi_cv']['accuracy']:.2%}",
        f"{agg_metrics['full_multi_task']['accuracy']:.2%}",
        f"{(agg_metrics['full_multi_cv']['accuracy'] - agg_metrics['full_multi_task']['accuracy']):.2%}",
        f"{multi_within:.1f}%",
        f"{conf_results['Multi_Rest']['ece']:.4f}",
        f"{conf_results['Multi_Rest']['high_conf_incorrect']:.1f}%",
        f"{binary_full_multi_cv['auc_roc'].mean():.4f}"
    ],
    'Interpretation': [
        'Higher = better',
        'Higher = better', 
        'Lower = more robust',
        'Higher = better localization',
        'Lower = better calibrated',
        'Lower = fewer confident mistakes',
        'Higher = better discrimination'
    ]
})

print("\nSUMMARY TABLE FOR THESIS:")
print(summary_df.to_string(index=False))


SUMMARY TABLE FOR THESIS:
                Metric    OvR Multinomial                   Interpretation
           CV Accuracy 93.17%      92.41%                  Higher = better
         Task Accuracy 89.63%      89.24%                  Higher = better
    Generalization Gap  3.54%       3.17%              Lower = more robust
 Within-Network Errors  12.4%       15.0%     Higher = better localization
                   ECE 0.2061      0.0913        Lower = better calibrated
High-Confidence Errors  32.0%       23.2% Lower = fewer confident mistakes
   Mean Per-Region AUC 0.9983      0.9993   Higher = better discrimination


In [35]:
# =============================================================================
# FINAL CONCLUSIONS
# =============================================================================

print("="*120)
print("CONCLUSIONS")
print("="*120)

print("""
Based on this comprehensive One-vs-Rest classification analysis:

1. METHODOLOGICAL CONTRIBUTION
   OvR classification addresses the "black box" limitation of multinomial regression by
   decomposing the 232-class problem into independent binary decisions. This enables:
   • Region-specific sensitivity and specificity metrics
   • Identification of "source" and "sink" hubs of confusion
   • Per-region generalization gap quantification

2. PERFORMANCE EQUIVALENCE
   OvR and multinomial achieve comparable overall accuracy, validating that the
   decomposition does not sacrifice classification performance.

3. ERROR PATTERN INSIGHTS
   Error taxonomy analysis reveals whether OvR produces more interpretable errors
   that respect functional network boundaries.

4. TASK-INDUCED REORGANIZATION
   The generalization gap analysis identifies specific networks and regions showing
   functional reorganization during Gender Stroop task engagement, supporting the
   central "error-as-signal" thesis framework.

5. IMPLICATIONS
   • Control and Attention networks show expected task sensitivity
   • Subcortical regions demonstrate distinct classification patterns
   • Per-region AUC provides a continuous measure of fingerprint distinctiveness

This analysis establishes OvR as a complementary approach to multinomial classification,
particularly valuable for interpreting classification errors as signals of functional
brain organization and task-induced neural dynamics.
""")

print("="*120)

CONCLUSIONS

Based on this comprehensive One-vs-Rest classification analysis:

1. METHODOLOGICAL CONTRIBUTION
   OvR classification addresses the "black box" limitation of multinomial regression by
   decomposing the 232-class problem into independent binary decisions. This enables:
   • Region-specific sensitivity and specificity metrics
   • Identification of "source" and "sink" hubs of confusion
   • Per-region generalization gap quantification

2. PERFORMANCE EQUIVALENCE
   OvR and multinomial achieve comparable overall accuracy, validating that the
   decomposition does not sacrifice classification performance.

3. ERROR PATTERN INSIGHTS
   Error taxonomy analysis reveals whether OvR produces more interpretable errors
   that respect functional network boundaries.

4. TASK-INDUCED REORGANIZATION
   The generalization gap analysis identifies specific networks and regions showing
   functional reorganization during Gender Stroop task engagement, supporting the
   central "error-as-s